# N-RDD2024 — RT-DETR-L Fine-Tune (Phase 2 — 70 epochs)

**Dataset:** N-RDD2024 (Kaya & Çodur, `doi:10.17632/27c8pwsd6v.3`)  
**Model:** RT-DETR-L  
**Accelerator:** GPU T4 x2  
**Classes:** 10 canonical classes (D00–D90)  

## This notebook runs Phase 2 only
Phase 1 (10 frozen epochs) already completed. `phase1_last.pt` must be
uploaded as a Kaggle dataset and added before running.

## Required Kaggle datasets
1. `sannyshankaranml/n-rdd2024` — the training data
2. `paraschiv/nrdd2024-phase1-weights` — upload your `last.pt` renamed as
   `phase1_last.pt` to this dataset before running

## Auto-resume
If the session times out mid-Phase 2, re-run all cells — Cell 8 detects
`last.pt` and resumes with `resume=True` from the exact interrupted epoch.


---
## Cell 1 — Install dependencies

`ultralytics==8.3.143` is installed. Does not touch Pillow or PyTorch.


In [ ]:
!pip install -q ultralytics==8.3.143 pyyaml tqdm psutil py-cpuinfo

import torch, ultralytics, PIL
print(f'PyTorch     : {torch.__version__}')
print(f'Ultralytics : {ultralytics.__version__}')
print(f'Pillow      : {PIL.__version__}')
print('Cell 1 complete.')


---
## Cell 2 — Environment verify and GPU check


In [ ]:
import os, torch, ultralytics
import ultralytics.utils.checks as _uc

# Patch check_amp: Ultralytics runs a yolo probe before training.
# Patching to True skips it. AMP still works during RT-DETR training.
_uc.check_amp = lambda model: True

if not torch.cuda.is_available():
    raise RuntimeError('No GPU — Kaggle → Settings → Accelerator → GPU T4 x2')

n_gpus = torch.cuda.device_count()
print(f'GPUs: {n_gpus}')
for i in range(n_gpus):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  CC={p.major}.{p.minor}  VRAM={p.total_memory/1024**3:.0f}GB')

for i in range(n_gpus):
    _ = torch.tensor([1.0, 2.0], device=f'cuda:{i}').float() / 255
    print(f'  [OK] CUDA tensor op on GPU {i}')

torch.backends.cudnn.benchmark     = True
torch.backends.cudnn.deterministic = False
print('Environment ready.')


---
## Cell 3 — Configuration


In [ ]:
from pathlib import Path
import torch

# ── Dataset paths ─────────────────────────────────────────────────────────
NRDD_ROOT = Path('/kaggle/input/datasets/sannyshankaranml/n-rdd2024')
TV_ROOT   = NRDD_ROOT / 'N-RDD2024Road damage and defects' / 'Training and Validation Dataset'
WORKING   = Path('/kaggle/working')

# ── Phase 1 weights dataset path ──────────────────────────────────────────
# Upload your last.pt (renamed to phase1_last.pt) to a Kaggle dataset.
# Adjust the path below to match your dataset slug.
PHASE1_WEIGHTS_DIR = Path('/kaggle/input/datasets/paraschiv/nrdd2024-phase1-weights')

PRETRAINED = 'rtdetr-l.pt'

# ── Canonical 10-class schema ──────────────────────────────────────────────
CANONICAL_DCODES = ['D00','D10','D20','D30','D40','D50','D60','D70','D80','D90']
CANONICAL_NAMES  = [
    'longitudinal_crack',        # ID 0 — D00
    'transverse_crack',          # ID 1 — D10
    'alligator_crack',           # ID 2 — D20
    'repaired_crack',            # ID 3 — D30
    'pothole',                   # ID 4 — D40
    'pedestrian_crossing_blur',  # ID 5 — D50
    'lane_line_blur',            # ID 6 — D60
    'manhole_cover',             # ID 7 — D70
    'patchy_road',               # ID 8 — D80
    'rutting',                   # ID 9 — D90
]
NC            = 10
CANONICAL_MAP = {d: i for i, d in enumerate(CANONICAL_DCODES)}

OVERSAMPLE = {9: 10, 3: 4, 8: 2, 5: 2}

COUNTRY_DIRS = [
    'Czech Republic_txt', 'USA_txt', 'china-motorbike_txt',
    'india_txt', 'japan_txt', 'norway_txt',
]

# ── Hyperparameters ────────────────────────────────────────────────────────
HP = dict(
    lr0=4.47e-4, lrf=0.01, momentum=0.9, weight_decay=5.27e-4,
    warmup_epochs=1, mosaic=0.860, mixup=0.205, copy_paste=0.0,
    degrees=5.0, translate=0.1, scale=0.5, shear=2.0,
    perspective=0.0, flipud=0.0, fliplr=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    box=7.685,
    cls=0.8,
    dfl=1.5,
    label_smoothing=0.1,
)

# ── Training schedule ──────────────────────────────────────────────────────
FREEZE_EPOCHS   = 10   # already done in Phase 1
FINETUNE_EPOCHS = 70   # Phase 2
IMGSZ           = 640
PATIENCE        = 20
SWA_N           = 5
SAVE_PERIOD     = 5

# ── Device and batch ──────────────────────────────────────────────────────
DEVICE  = '0,1' if torch.cuda.device_count() > 1 else '0'
WORKERS = 4
vram    = torch.cuda.get_device_properties(0).total_memory / 1024**3
BATCH   = 16 if vram >= 12.0 else (8 if vram >= 6.0 else 4)

# ── Run paths ─────────────────────────────────────────────────────────────
RUN_NAME     = 'rtdetr_nrdd2024'
OUTPUT_DIR   = WORKING / 'runs' / 'detect'
RUN_DIR      = OUTPUT_DIR / RUN_NAME
DATASET_DIR  = WORKING / 'dataset'
DATASET_YAML = WORKING / 'dataset_nrdd2024.yaml'

print(f'TV_ROOT exists         : {TV_ROOT.exists()}')
print(f'Phase1 weights exists  : {PHASE1_WEIGHTS_DIR.exists()}')
print(f'VRAM: {vram:.0f} GB  batch={BATCH}  device={DEVICE}')
print(f'Phase 2 epochs: {FINETUNE_EPOCHS}')


---
## Cell 4 — Per-country class ID remap tables


In [ ]:
import yaml

country_info = {}
errors = []

print('=== Per-country class ID remap tables ===')
for country in COUNTRY_DIRS:
    country_root = TV_ROOT / country
    if not country_root.exists():
        errors.append(f'[ERROR] Not found: {country_root}'); continue
    yaml_path = country_root / 'data.yaml'
    if not yaml_path.exists(): yaml_path = country_root / 'train' / 'data.yaml'
    if not yaml_path.exists():
        errors.append(f'[ERROR] data.yaml not found in {country_root}'); continue
    with open(yaml_path) as f: cfg = yaml.safe_load(f)
    local_names = cfg.get('names', [])
    nc_local    = cfg.get('nc', len(local_names))
    remap = {}
    for local_id, dcode in enumerate(local_names):
        if dcode in CANONICAL_MAP: remap[local_id] = CANONICAL_MAP[dcode]
        else: errors.append(f'[WARN] {country}: unknown D-code "{dcode}"')
    country_info[country] = {'root': country_root, 'local_names': local_names,
                              'nc': nc_local, 'remap': remap}
    print(f'{country}  (nc={nc_local})')
    for lid, dcode in enumerate(local_names):
        cid = CANONICAL_MAP.get(dcode, '?')
        eng = CANONICAL_NAMES[cid] if isinstance(cid, int) else 'UNKNOWN'
        print(f'  {lid} {dcode} -> {cid} {eng}')

if errors:
    for e in errors: print(e)
    if any(e.startswith('[ERROR]') for e in errors):
        raise FileNotFoundError('Critical path errors — fix COUNTRY_DIRS or TV_ROOT.')
print(f'Countries loaded: {list(country_info.keys())}')


---
## Cell 5 — Remap label files and build dataset directory


In [ ]:
import os, shutil
from collections import defaultdict
from tqdm import tqdm

image_class_index = {'train': defaultdict(list), 'valid': defaultdict(list)}
all_pairs = {'train': [], 'valid': []}
total_remapped = total_skipped = total_images = 0

print('=== Remapping labels and building dataset structure ===')
for country, info in country_info.items():
    remap = info['remap']; root = info['root']
    for split in ['train', 'valid']:
        img_dir   = root / split / 'images'
        label_dir = root / split / 'labels'
        if not img_dir.exists() or not label_dir.exists():
            print(f'  [SKIP] {country}/{split}'); continue
        out_img   = DATASET_DIR / split / 'images'
        out_label = DATASET_DIR / split / 'labels'
        out_img.mkdir(parents=True, exist_ok=True)
        out_label.mkdir(parents=True, exist_ok=True)
        n_rem = n_skip = n_img = 0
        for lp in tqdm(sorted(label_dir.glob('*.txt')), desc=f'  {country}/{split}', leave=False):
            ip = None
            for ext in ['.jpg','.jpeg','.png']:
                c = img_dir / (lp.stem + ext)
                if c.exists(): ip = c; break
            if ip is None: continue
            raw = lp.read_text().strip(); lines = []; cids = set()
            if raw:
                for line in raw.splitlines():
                    parts = line.strip().split()
                    if len(parts) < 5: continue
                    try: lid = int(parts[0])
                    except ValueError: n_skip += 1; continue
                    if lid not in remap: n_skip += 1; continue
                    cid = remap[lid]
                    lines.append(f"{cid} {' '.join(parts[1:])}"); cids.add(cid); n_rem += 1
            ol = out_label / lp.name; ol.write_text('\n'.join(lines))
            lk = out_img / ip.name
            if not lk.exists():
                try: os.symlink(ip.resolve(), lk)
                except OSError:
                    try: os.link(ip.resolve(), lk)
                    except OSError: shutil.copy2(ip, lk)
            pair = (str(lk), str(ol))
            all_pairs[split].append(pair)
            for cid in cids: image_class_index[split][cid].append(pair)
            n_img += 1
        print(f'  {country:<25} {split:<6}  images:{n_img:>7,}  remapped:{n_rem:>9,}  skipped:{n_skip}')
        total_remapped += n_rem; total_skipped += n_skip; total_images += n_img

print(f'Total images: {total_images:,}  Remapped: {total_remapped:,}  Skipped: {total_skipped}')
print('[OK]' if total_skipped == 0 else '[WARN] Check remap tables in Cell 4.')


---
## Cell 6 — Class-imbalance oversampling + dataset.yaml


In [ ]:
import os, shutil, yaml, random
from collections import Counter
from pathlib import Path

train_pairs = list(all_pairs['train'])
for can_id, factor in OVERSAMPLE.items():
    rare = image_class_index['train'][can_id]
    if not rare: print(f'[WARN] No images for canonical ID {can_id}'); continue
    extra = rare * (factor - 1); train_pairs.extend(extra)
    print(f'  Oversampled {CANONICAL_NAMES[can_id]:<28} {len(rare):>5,} x{factor} -> +{len(extra):,}')

random.seed(42); random.shuffle(train_pairs)
print(f'Train after oversampling : {len(train_pairs):,}')
print(f'Valid (not oversampled)  : {len(all_pairs["valid"]):,}')

OVER_IMG   = DATASET_DIR / 'train_oversampled' / 'images'
OVER_LABEL = DATASET_DIR / 'train_oversampled' / 'labels'
OVER_IMG.mkdir(parents=True, exist_ok=True)
OVER_LABEL.mkdir(parents=True, exist_ok=True)

seen = Counter()
for ip_str, lp_str in train_pairs:
    ip = Path(ip_str); lp = Path(lp_str)
    seen[ip.stem] += 1; n = seen[ip.stem]
    new_stem = ip.stem if n == 1 else f'{ip.stem}_dup{n}'
    ni = OVER_IMG   / (new_stem + ip.suffix)
    nl = OVER_LABEL / (new_stem + '.txt')
    if not ni.exists():
        try: os.symlink(ip.resolve(), ni)
        except OSError:
            try: os.link(ip.resolve(), ni)
            except OSError: shutil.copy2(ip, ni)
    if not nl.exists(): shutil.copy2(lp, nl)

print(f'Oversampled train dir: {len(list(OVER_IMG.glob("*"))):,} images')

names_dict = {i: n for i, n in enumerate(CANONICAL_NAMES)}
header = (
    '# N-RDD2024 dataset.yaml\n'
    '# Source: Kaya & Codur, doi:10.17632/27c8pwsd6v.3\n'
    '# ID  D-code  English name\n'
    '#  0  D00     longitudinal_crack\n'
    '#  1  D10     transverse_crack\n'
    '#  2  D20     alligator_crack\n'
    '#  3  D30     repaired_crack\n'
    '#  4  D40     pothole\n'
    '#  5  D50     pedestrian_crossing_blur\n'
    '#  6  D60     lane_line_blur\n'
    '#  7  D70     manhole_cover\n'
    '#  8  D80     patchy_road\n'
    '#  9  D90     rutting\n'
)
with open(DATASET_YAML, 'w') as f:
    f.write(header)
    yaml.dump({'path': str(DATASET_DIR), 'train': 'train_oversampled/images',
               'val': 'valid/images', 'nc': NC, 'names': names_dict},
              f, default_flow_style=False, sort_keys=False)
print(f'dataset.yaml -> {DATASET_YAML}')
print(DATASET_YAML.read_text())


---
## Cell 7 — Post-remap class distribution check


In [ ]:
from collections import Counter

cc = Counter(); empty = 0
for lp in (DATASET_DIR / 'train' / 'labels').glob('*.txt'):
    t = lp.read_text().strip()
    if not t: empty += 1; continue
    for line in t.splitlines():
        p = line.strip().split()
        if p:
            try: cc[int(p[0])] += 1
            except ValueError: pass

total = sum(cc.values())
print(f'Train label files : {len(list((DATASET_DIR/"train"/"labels").glob("*.txt"))):,}')
print(f'Empty (hard negs) : {empty:,}')
print(f'Total annotations : {total:,}')
print()
print(f'  {"ID":<4} {"D-code":<6} {"English name":<28} {"Count":>8} {"Share":>7}')
print('  ' + '-'*58)
for i,(d,n) in enumerate(zip(CANONICAL_DCODES, CANONICAL_NAMES)):
    c = cc.get(i,0); pct = c/total*100 if total else 0
    flag = '  <- zero' if c==0 else ''
    print(f'  {i:<4} {d:<6} {n:<28} {c:>8,} {pct:>6.1f}%{flag}')


---
## Cell 8 — Phase 2: Full fine-tune (70 epochs) + auto-resume

**Resume logic:**
1. Finds `last.pt` in run/weights/ → `resume=True` (continues interrupted Phase 2)
2. Finds `phase1_last.pt` in /kaggle/working/ → `resume=False` (starts Phase 2 fresh)
3. Neither found → looks in the Phase 1 weights dataset → starts Phase 2 fresh
4. Nothing found anywhere → error (Phase 1 must be done first)

Persistence watcher saves `best.pt`, `last.pt`, `results.csv` every 5 minutes.


In [ ]:
import shutil, threading
from pathlib import Path
from ultralytics import RTDETR

# ── Shared training kwargs ─────────────────────────────────────────────────
SHARED = dict(
    data=str(DATASET_YAML), imgsz=IMGSZ, batch=BATCH, workers=WORKERS,
    device=DEVICE, optimizer='AdamW', cos_lr=True, amp=True, cache=False,
    plots=True, save=True, save_period=SAVE_PERIOD, val=True, verbose=True,
    deterministic=False, project=str(OUTPUT_DIR), name=RUN_NAME, exist_ok=True,
    weight_decay=HP['weight_decay'], momentum=HP['momentum'], lrf=HP['lrf'],
    mosaic=HP['mosaic'], mixup=HP['mixup'], copy_paste=HP['copy_paste'],
    degrees=HP['degrees'], translate=HP['translate'], scale=HP['scale'],
    shear=HP['shear'], perspective=HP['perspective'], flipud=HP['flipud'],
    fliplr=HP['fliplr'], hsv_h=HP['hsv_h'], hsv_s=HP['hsv_s'], hsv_v=HP['hsv_v'],
    box=HP['box'], cls=HP['cls'], dfl=HP['dfl'],
)

# ── Checkpoint paths ───────────────────────────────────────────────────────
weights_dir     = RUN_DIR / 'weights'
last_pt_run     = weights_dir / 'last.pt'
best_pt_run     = weights_dir / 'best.pt'
best_pt_working = WORKING / 'best.pt'
phase1_backup   = WORKING / 'phase1_last.pt'
phase2_start    = WORKING / 'phase2_start.pt'

# ── Step 1: Restore from working backup if run/ was wiped ─────────────────
# Happens when Kaggle restarts the session — runs/ is cleared but /kaggle/working/ persists.
if best_pt_working.exists() and not best_pt_run.exists():
    weights_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(best_pt_working, best_pt_run)
    print('Restored best.pt from working backup.')

# ── Step 2: Restore phase1_last.pt from dataset if not in working ──────────
# This is the case where you have Phase 1 weights uploaded as a dataset.
dataset_phase1 = PHASE1_WEIGHTS_DIR / 'phase1_last.pt'
if not phase1_backup.exists() and not last_pt_run.exists() and dataset_phase1.exists():
    shutil.copy2(dataset_phase1, phase1_backup)
    print(f'Restored phase1_last.pt from dataset: {dataset_phase1}')

# ── Step 3: Determine training mode ───────────────────────────────────────
# Priority:
#   a) last.pt in run/weights/ -> resume interrupted Phase 2 (resume=True)
#   b) phase1_last.pt in /kaggle/working/ -> start Phase 2 fresh (resume=False)
#   c) Nothing -> error
if last_pt_run.exists():
    # Phase 2 was already started and interrupted — resume it.
    # Copy to /kaggle/working/ so DDP subprocesses can reliably find it.
    shutil.copy2(last_pt_run, phase2_start)
    resume_ckpt   = str(phase2_start)
    training_mode = 'resume_phase2'
    is_resuming   = True
elif phase1_backup.exists():
    # Phase 1 done, Phase 2 not yet started — start fresh.
    # Copy to /kaggle/working/ so DDP subprocesses can reliably find it.
    shutil.copy2(phase1_backup, phase2_start)
    resume_ckpt   = str(phase2_start)
    training_mode = 'start_phase2'
    is_resuming   = False
else:
    raise FileNotFoundError(
        'No Phase 1 checkpoint found.\n'
        'Upload last.pt (renamed to phase1_last.pt) to the '
        'paraschiv/nrdd2024-phase1-weights dataset and add it to this notebook.'
    )

print(f'Training mode : {training_mode}')
print(f'Resume ckpt   : {resume_ckpt}')
print(f'Resuming      : {is_resuming}')
print(f'Device        : {DEVICE}   Batch: {BATCH}')

# ── Persistence watcher ────────────────────────────────────────────────────
# Copies best.pt, last.pt, results.csv to /kaggle/working/ every 5 minutes.
# This ensures outputs survive session timeouts.
_stop = threading.Event()
def _watch(interval=300):
    while not _stop.wait(timeout=interval):
        for fn in ['best.pt', 'last.pt']:
            s = weights_dir / fn
            if s.exists(): shutil.copy2(s, WORKING / fn)
        s = RUN_DIR / 'results.csv'
        if s.exists(): shutil.copy2(s, WORKING / 'results.csv')
threading.Thread(target=_watch, daemon=True).start()
print('Persistence watcher started (saves every 5 min).')

# ── Phase 2: Full fine-tune ────────────────────────────────────────────────
print(f'\n-- Phase 2: Full fine-tune ({FINETUNE_EPOCHS} epochs) --')
print(f'  Start    : {resume_ckpt}')
print(f'  LR       : {HP["lr0"]*0.1}   Patience: {PATIENCE}')
print(f'  Resuming : {is_resuming}')

model_p2 = RTDETR(resume_ckpt)
model_p2.train(
    **SHARED,
    epochs=FINETUNE_EPOCHS, lr0=HP['lr0']*0.1,
    warmup_epochs=1, freeze=0, patience=PATIENCE, resume=is_resuming,
)

# Final safety copy
for fn in ['best.pt', 'last.pt']:
    s = weights_dir / fn
    if s.exists():
        shutil.copy2(s, WORKING / fn)
        print(f'Saved {fn} -> {WORKING / fn}')

_stop.set()
print('\nPhase 2 complete. Proceed to Cell 9 (SWA).')


---
## Cell 9 — SWA (Stochastic Weight Averaging)

Averages the last `SWA_N` epoch checkpoints.  
Reference: Izmailov et al. 2018, `arxiv.org/abs/1803.05407`.


In [ ]:
import torch, shutil

def apply_swa(run_dir, swa_n=5):
    wd    = run_dir / 'weights'
    ckpts = sorted(wd.glob('epoch*.pt'))
    if len(ckpts) < 2:
        print(f'[SWA] Only {len(ckpts)} epoch checkpoints -- skipping.'); return None
    sel = ckpts[-swa_n:]
    print(f'[SWA] Averaging: {[c.name for c in sel]}')
    base = torch.load(sel[0], map_location='cpu', weights_only=False)
    bsd  = base['model'].state_dict() if hasattr(base.get('model',None),'state_dict') else base['model']
    avg  = {k: v.clone().float() for k,v in bsd.items()}
    for p in sel[1:]:
        sd = torch.load(p, map_location='cpu', weights_only=False)
        sd = sd['model'].state_dict() if hasattr(sd.get('model',None),'state_dict') else sd['model']
        for k in avg:
            if k in sd: avg[k] += sd[k].float()
    for k in avg: avg[k] /= len(sel)
    best = torch.load(wd/'best.pt', map_location='cpu', weights_only=False)
    if hasattr(best.get('model',None),'load_state_dict'):
        best['model'].load_state_dict({k: v.half() for k,v in avg.items()})
    else:
        best['model'] = {k: v.half() for k,v in avg.items()}
    out = wd / 'swa.pt'; torch.save(best, out)
    print(f'[SWA] -> {out}'); return out

swa = apply_swa(RUN_DIR, SWA_N)
if swa and swa.exists():
    shutil.copy2(swa, WORKING/'swa.pt')
    print(f'swa.pt -> {WORKING/"swa.pt"}')


---
## Cell 10 — Full metrics evaluation and plots

Formal validation at `conf=0.001, iou=0.6` (COCO mAP protocol).


In [ ]:
import shutil, pandas as pd
import matplotlib.pyplot as plt, matplotlib.image as mpimg
from ultralytics import RTDETR

wd = RUN_DIR / 'weights'
eval_ckpt = next((wd/f for f in ['swa.pt','best.pt','last.pt'] if (wd/f).exists()), None)
if eval_ckpt is None: raise FileNotFoundError('No checkpoint found. Run Cell 8 first.')
print(f'Evaluating: {eval_ckpt}')

model_eval = RTDETR(str(eval_ckpt))
metrics = model_eval.val(
    data=str(DATASET_YAML), imgsz=IMGSZ, batch=BATCH, device=DEVICE,
    conf=0.001, iou=0.6, plots=True, verbose=True,
)

print('\n' + '='*60)
print('EVALUATION RESULTS -- N-RDD2024 Fine-Tune')
print('='*60)
try:
    print(f'  mAP50     : {metrics.box.map50:.4f}')
    print(f'  mAP50-95  : {metrics.box.map:.4f}')
    print(f'  Precision : {metrics.box.mp:.4f}')
    print(f'  Recall    : {metrics.box.mr:.4f}')
    f1 = 2*metrics.box.mp*metrics.box.mr/(metrics.box.mp+metrics.box.mr+1e-9)
    print(f'  F1        : {f1:.4f}')
    print()
    print(f'  {"ID":<4} {"D-code":<6} {"English name":<28} {"AP@50":>8} {"AP@50:95":>10}')
    print('  '+'-'*60)
    for i,(d,n) in enumerate(zip(CANONICAL_DCODES,CANONICAL_NAMES)):
        ap50   = metrics.box.ap50[i] if i<len(metrics.box.ap50) else float('nan')
        ap5095 = metrics.box.ap[i]   if i<len(metrics.box.ap)   else float('nan')
        print(f'  {i:<4} {d:<6} {n:<28} {ap50:>8.4f} {ap5095:>10.4f}')
except Exception as e:
    print(f'[Metrics object error: {e}] -- check results.csv below.')

# Training curves
results_csv = RUN_DIR / 'results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv); df.columns = df.columns.str.strip()
    map50_cols   = [c for c in df.columns if 'mAP50' in c and '95' not in c]
    map5095_cols = [c for c in df.columns if 'mAP50-95' in c or 'mAP50_95' in c]
    prec_cols    = [c for c in df.columns if 'precision' in c.lower()]
    rec_cols     = [c for c in df.columns if 'recall' in c.lower()]
    tl_cols      = [c for c in df.columns if 'train' in c.lower() and 'loss' in c.lower()]
    vl_cols      = [c for c in df.columns if 'val' in c.lower() and 'loss' in c.lower()]
    ep = df.columns[0]
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    axes = axes.flatten()
    for ax,(cols,title) in zip(axes,[
        (map50_cols,'mAP@0.50'),(map5095_cols,'mAP@0.50:0.95'),
        (prec_cols,'Precision'),(rec_cols,'Recall'),
        (tl_cols,'Train Loss'),(vl_cols,'Val Loss')]):
        if cols:
            ax.plot(df[ep], df[cols[0]], linewidth=2)
            ax.set_title(title, fontweight='bold'); ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
    plt.suptitle('N-RDD2024 RT-DETR-L Training Curves', fontsize=14, fontweight='bold')
    plt.tight_layout()
    cpath = WORKING/'training_curves.png'
    plt.savefig(cpath, dpi=150, bbox_inches='tight'); plt.show()
    if map50_cols:
        best = df.loc[df[map50_cols[0]].idxmax()]
        print(f'Best epoch mAP50={best[map50_cols[0]]:.4f}')

# Ultralytics plots
for pp, title in [
    (RUN_DIR/'PR_curve.png','Precision-Recall Curve'),
    (RUN_DIR/'F1_curve.png','F1-Confidence Curve'),
    (RUN_DIR/'P_curve.png','Precision-Confidence Curve'),
    (RUN_DIR/'R_curve.png','Recall-Confidence Curve'),
    (RUN_DIR/'confusion_matrix_normalized.png','Confusion Matrix (normalised)'),
    (RUN_DIR/'confusion_matrix.png','Confusion Matrix (raw)'),
    (RUN_DIR/'labels.jpg','Label Distribution'),
]:
    if pp.exists():
        fig, ax = plt.subplots(figsize=(12,8))
        ax.imshow(mpimg.imread(str(pp))); ax.axis('off')
        ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
        plt.tight_layout(); plt.show()
        shutil.copy2(pp, WORKING/pp.name)


---
## Cell 11 — Final checkpoint packaging


In [ ]:
import shutil

wd = RUN_DIR / 'weights'
copies = {
    'best.pt': WORKING/'rtdetr_l_nrdd2024_best.pt',
    'last.pt': WORKING/'rtdetr_l_nrdd2024_last.pt',
    'swa.pt' : WORKING/'rtdetr_l_nrdd2024_swa.pt',
}
print('=== Output weights ===')
for src_name, dst in copies.items():
    src = wd / src_name
    if src.exists():
        shutil.copy2(src, dst)
        print(f'  {dst.name:<45} {dst.stat().st_size/1024**2:.1f} MB')
    else:
        print(f'  [MISSING] {src_name}')
for fn in ['results.csv', 'args.yaml', 'training_curves.png']:
    src = RUN_DIR/fn if fn != 'training_curves.png' else WORKING/fn
    if src.exists(): shutil.copy2(src, WORKING/fn); print(f'  {fn}')
shutil.copy2(DATASET_YAML, WORKING/'dataset_nrdd2024.yaml')
print('\n=== All files in /kaggle/working/ ===')
for f in sorted(WORKING.glob('*')):
    if f.is_file():
        print(f'  {f.name:<45} {f.stat().st_size/1024**2:.1f} MB')
print('\nDone. Download rtdetr_l_nrdd2024_best.pt from the Output tab.')
